# 🧹 Pure Data Retraining v5 - Drop Toxic, No Constraints

**Strategy:** Instead of fixing toxic rows to 0.0, DROP them completely.

**Philosophy:**
- Let the model learn ONLY from clean, clinically valid examples
- No monotonic constraints - trust the pure data to teach patterns naturally
- Early stopping as the safety brake

**Output:** `xgboost_final_model_v5_pure.json`

## 1. Setup

In [ ]:
!pip install polars xgboost -q

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np
import os
import math

print(f"Polars: {pl.__version__}")
print(f"XGBoost: {xgb.__version__}")

In [ ]:
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data/processed"

FILES = {
    "train": f"{BASE_PATH}/train_augmented_fixed.parquet",
    "val": f"{BASE_PATH}/test_ready_for_xgboost.parquet",
    "output_model": f"{BASE_PATH}/xgboost_final_model_v5_pure.json",
    "output_train": f"{BASE_PATH}/train_pure_v5.parquet"
}

for name, path in FILES.items():
    if "output" not in name:
        exists = "✅" if os.path.exists(path) else "❌"
        print(f"{exists} {name}: {path}")

## 2. Define Toxic Mask

In [ ]:
def get_toxic_mask():
    """Define 4 toxic scenarios to DROP."""
    return (
        # 1. Falling & Low
        ((pl.col("CGM") < 110) & 
         (pl.col("cgm_delta_15min") < -5) & 
         (pl.col("carbs") == 0)) |
        
        # 2. Hypo Hazard
        ((pl.col("CGM") < 70) & 
         (pl.col("carbs") == 0)) |
        
        # 3. Rapid Drop
        ((pl.col("cgm_delta_15min") < -15) & 
         (pl.col("carbs") == 0)) |
        
        # 4. Stable Low
        ((pl.col("CGM") >= 70) & 
         (pl.col("CGM") < 100) & 
         (pl.col("cgm_delta_15min").abs() < 3) & 
         (pl.col("carbs") == 0))
    )

print("✅ Toxic mask defined (4 scenarios)")

## 3. Load & Filter Training Data

In [ ]:
print("📂 Loading training data...")
df_train_raw = pl.read_parquet(FILES["train"])
n_train_before = len(df_train_raw)
print(f"   Raw: {n_train_before:,} rows")

# Count toxic rows BEFORE dropping
n_toxic_train = df_train_raw.filter(get_toxic_mask()).shape[0]
n_toxic_with_bolus = df_train_raw.filter(get_toxic_mask() & (pl.col("bolus") > 0)).shape[0]

print(f"\n⚠️ Toxic Rows Found:")
print(f"   Total matching toxic scenarios: {n_toxic_train:,}")
print(f"   With bolus > 0 (truly dangerous): {n_toxic_with_bolus:,}")

In [ ]:
# DROP toxic rows (keep NOT toxic)
print("\n🧹 DROPPING toxic rows from TRAINING...")
df_train = df_train_raw.filter(~get_toxic_mask())
n_train_after = len(df_train)
n_dropped_train = n_train_before - n_train_after

print(f"   Before: {n_train_before:,}")
print(f"   After:  {n_train_after:,}")
print(f"   📉 Dropped: {n_dropped_train:,} rows ({n_dropped_train/n_train_before*100:.1f}%)")

In [ ]:
# Save pure training data
print(f"\n💾 Saving pure training data to: {FILES['output_train']}")
df_train.write_parquet(FILES['output_train'])
print("✅ Saved!")

## 4. Load & Filter Validation Data

In [ ]:
print("📂 Loading validation data...")
df_val_raw = pl.read_parquet(FILES["val"])
n_val_before = len(df_val_raw)
print(f"   Raw: {n_val_before:,} rows")

# DROP toxic rows from validation too
print("\n🧹 DROPPING toxic rows from VALIDATION...")
df_val = df_val_raw.filter(~get_toxic_mask())
n_val_after = len(df_val)
n_dropped_val = n_val_before - n_val_after

print(f"   Before: {n_val_before:,}")
print(f"   After:  {n_val_after:,}")
print(f"   📉 Dropped: {n_dropped_val:,} rows ({n_dropped_val/n_val_before*100:.1f}%)")

print(f"\n📊 TOTAL DROPPED: {n_dropped_train + n_dropped_val:,} rows")

## 5. Features & DMatrix

In [ ]:
FEATURES = [
    'CGM', 'carbs', 'time_sin', 'time_cos', 'is_weekend',
    'cgm_delta_5min', 'cgm_delta_15min', 'basal',
    'age', 'weight', 'gender_encoded', 'insulin_type_encoded'
]
TARGET = 'bolus'

available = [f for f in FEATURES if f in df_train.columns]
print(f"✅ Features: {len(available)}/{len(FEATURES)}")

# Prepare arrays
X_train = df_train.select(available).to_numpy().astype(np.float32)
y_train = df_train[TARGET].to_numpy().astype(np.float32)

X_val = df_val.select(available).to_numpy().astype(np.float32)
y_val = df_val[TARGET].to_numpy().astype(np.float32)

# DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=available)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=available)

print(f"\n✅ Train: {dtrain.num_row():,} × {dtrain.num_col()}")
print(f"✅ Val: {dval.num_row():,} × {dval.num_col()}")

# Class balance
n_zero = (y_train == 0).sum()
n_pos = (y_train > 0).sum()
print(f"\n📊 Balance: {n_zero:,} zeros / {n_pos:,} positive (ratio: {n_zero/n_pos:.2f})")

## 6. Pure XGBoost Config (No Constraints)

In [ ]:
PARAMS = {
    # Objective
    'objective': 'reg:squarederror',
    'eval_metric': 'mae',
    
    # GPU
    'tree_method': 'hist',
    'device': 'cuda',
    
    # NO monotone_constraints! Let clean data teach naturally
    
    # Neutral weight (data is clean now)
    'scale_pos_weight': 1.0,
    'max_delta_step': 1,
    
    # Tree
    'max_depth': 8,
    'learning_rate': 0.05,
    'min_child_weight': 3,
    
    # Sampling
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    
    # Regularization
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    
    'seed': 42,
    'verbosity': 1
}

NUM_ROUNDS = 1000
EARLY_STOPPING = 50

print("⚙️ PURE CONFIG (No Constraints):")
print(f"   Rounds: {NUM_ROUNDS} (Early Stop: {EARLY_STOPPING})")
print(f"   Learning Rate: {PARAMS['learning_rate']}")
print(f"   Max Depth: {PARAMS['max_depth']}")
print(f"   Monotonic Constraints: DISABLED ✗")

## 7. Train

In [ ]:
%%time

print("🚀 TRAINING (Pure Data Strategy)...")
print("=" * 50)

evals = [(dtrain, 'train'), (dval, 'val')]
evals_result = {}

model = xgb.train(
    params=PARAMS,
    dtrain=dtrain,
    num_boost_round=NUM_ROUNDS,
    evals=evals,
    evals_result=evals_result,
    early_stopping_rounds=EARLY_STOPPING,
    verbose_eval=100
)

print("\n" + "=" * 50)
print(f"✅ TRAINING COMPLETE!")
print(f"   Best Iteration: {model.best_iteration}")
print(f"   Best Val MAE: {model.best_score:.4f}")

In [ ]:
print(f"\n💾 Saving model to: {FILES['output_model']}")
model.save_model(FILES['output_model'])
print("✅ Model saved!")

## 8. Critical Safety Tests

In [ ]:
def create_test_case(cgm, carbs, delta_15, delta_5=None):
    if delta_5 is None:
        delta_5 = delta_15 / 3
    
    hour = 12
    features = [
        cgm, carbs,
        math.sin(2 * math.pi * hour / 24),
        math.cos(2 * math.pi * hour / 24),
        0.0,  # is_weekend
        delta_5, delta_15,
        0.8,  # basal
        35.0, 70.0,  # age, weight
        0.0, 3.0  # gender, insulin_type
    ]
    return np.array([features], dtype=np.float32)

def test_prediction(name, cgm, carbs, delta, max_expected):
    X = create_test_case(cgm, carbs, delta)
    dtest = xgb.DMatrix(X, feature_names=available)
    pred = model.predict(dtest)[0]
    
    passed = pred < max_expected
    status = "✅" if passed else "❌"
    
    print(f"{status} {name}")
    print(f"   CGM={cgm}, Carbs={carbs}, Delta={delta}")
    print(f"   Pred: {pred:.4f} U (Max: {max_expected})")
    return passed

In [ ]:
print("🧪 CRITICAL SAFETY TESTS")
print("=" * 50)

safety_tests = [
    ("Hypo No Food", 65, 0, 0, 0.1),
    ("Falling Trend No Food (88 mg/dL)", 88, 0, -10, 0.1),
    ("Rapid Drop", 120, 0, -20, 0.1),
    ("Stable Low", 85, 0, 0, 0.1),
]

results = [test_prediction(*t) for t in safety_tests]

print("\n" + "=" * 50)
passed = sum(results)
total = len(results)
print(f"SAFETY TESTS: {passed}/{total} passed")

if passed == total:
    print("🎉 ALL SAFETY TESTS PASSED!")
else:
    print("⚠️ SOME TESTS FAILED - Add safety rules in API")

In [ ]:
print("\n🧪 POSITIVE CASES (Should recommend insulin)")
print("=" * 50)

positive_tests = [
    ("High + Carbs", 200, 60, 5, 0.5),
    ("Very High", 280, 30, 10, 0.3),
    ("Rising + Meal", 150, 45, 15, 0.2),
]

for name, cgm, carbs, delta, min_exp in positive_tests:
    X = create_test_case(cgm, carbs, delta)
    dtest = xgb.DMatrix(X, feature_names=available)
    pred = model.predict(dtest)[0]
    status = "✅" if pred >= min_exp else "⚠️"
    print(f"{status} {name}: {pred:.2f} U (min: {min_exp})")

## 9. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importance = model.get_score(importance_type='gain')
imp_df = pl.DataFrame({
    'feature': list(importance.keys()),
    'importance': list(importance.values())
}).sort('importance', descending=True)

print("📊 Feature Importance:")
print(imp_df)

plt.figure(figsize=(10, 5))
plt.barh(imp_df['feature'].to_list()[::-1], 
         imp_df['importance'].to_list()[::-1])
plt.xlabel('Gain')
plt.title('Feature Importance - Pure Data Model v5')
plt.tight_layout()
plt.show()

## 10. Summary

In [ ]:
print("\n" + "=" * 60)
print("📋 PURE DATA TRAINING SUMMARY")
print("=" * 60)
print(f"""
Data Filtering (DROP, not fix):
   - Training rows dropped: {n_dropped_train:,}
   - Validation rows dropped: {n_dropped_val:,}
   - Total dropped: {n_dropped_train + n_dropped_val:,}

Model:
   - Strategy: Pure Data (No Constraints)
   - Best Iteration: {model.best_iteration}
   - Best Val MAE: {model.best_score:.4f}

Output:
   - Model: {FILES['output_model']}
   - Data: {FILES['output_train']}

Safety Tests: {passed}/{total} passed
""")
print("=" * 60)